# 02 · Trigger Log Deep Inspection

**Project:** Physics-Informed Trigger Event Analysis  
**Dataset:** USDOT ISC Stage-1B  
**Stage:** Task 2 — Trigger semantics (no ML, no extraction)

---

## Purpose

Deeply inspect `Radars_Run_XXXX_traffic-triggers-output.json` files to:
- Confirm top-level MQTT structure.
- Find `reference_name`, `associated_lane`, `associated_zone`, `associated_sensor` inside `payload`.
- Build a `reference_name → lane / zone / sensor` mapping table.
- Compare trigger lane/zone identifiers with radar object `closest_lane` / `within_zone`.
- Probe timestamp alignment between radar `receivedAt` and trigger `receivedAt`.

> **Semantic warning:** `reference_name` values are treated as detector/zone
> activation identifiers until confirmed otherwise by cross-modal evidence.

## Outputs

| File | Description |
|------|-------------|
| `trigger_file_inventory.csv` | All trigger JSON paths across all 4 zips |
| `trigger_nested_field_paths.csv` | Where semantic fields live inside payload |
| `sample_trigger_event_table.csv` | One row per trigger event from sampled runs |
| `sample_trigger_reference_mapping.csv` | `reference_name → lane/zone/sensor` counts |
| `trigger_topic_samples.csv` | Unique MQTT topic strings with parsed components |
| `sample_trigger_frequency_summary.csv` | Record/event counts per sampled run |
| `sample_radar_trigger_lane_zone_comparison.csv` | Radar vs trigger lane/zone values |
| `sample_trigger_radar_alignment_probe.csv` | Timestamp gap between radar and trigger |
| `notebook_02_trigger_findings.md` | Auto-generated findings summary |

---
## 0 · Optional: Install / Verify Dependencies

In [1]:
# !pip install -q pandas
print("[OK] Dependency check cell.")

[OK] Dependency check cell.


---
## 1 · Clone Repo from GitHub

**Edit `GITHUB_REPO_URL` before running.**

In [2]:
import os, subprocess

# ╔══════════════════════════════════════════════════════════════════╗
# ║  EDIT THIS URL                                                 ║
# ╠══════════════════════════════════════════════════════════════════╣
GITHUB_REPO_URL = "https://github.com/PulockDas/intersection_safety_trigger_project.git"
# ╚══════════════════════════════════════════════════════════════════╝

CLONE_DIR = "/content/intersection_safety_trigger_project"
if not os.path.exists(CLONE_DIR):
    result = subprocess.run(
        ["git", "clone", GITHUB_REPO_URL, CLONE_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"Git clone failed:\n{result.stderr}")
    print("[OK] Clone successful.")
else:
    print(f"[INFO] Repo already at {CLONE_DIR}.")
print("Contents:", os.listdir(CLONE_DIR))

[OK] Clone successful.
Contents: ['requirements.txt', '.git', 'README.md', 'notebooks', 'outputs', 'src', '.gitignore']


### 1.1 · Mount Google Drive (read-only — training zip files)

In [3]:
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
    print("[INFO] Drive mounted — READ ONLY for zip files.")
except ImportError:
    print("[INFO] Not in Colab — Drive mount skipped.")

Mounted at /content/drive
[INFO] Drive mounted — READ ONLY for zip files.


---
## 2 · Project Setup

Run **Cell 2.0** to auto-detect the training zip folder, then **Cell 2.1** to set paths.

### 2.0 · Locate the Training Data Folder

In [4]:
import os
from pathlib import Path

# ╔══════════════════════════════════════════════════════════════════╗
FOLDER_NAME = "ITS_Intersection_USDOT"
# ╚══════════════════════════════════════════════════════════════════╝

DRIVE_ROOT   = Path("/content/drive")
MY_DRIVE     = DRIVE_ROOT / "MyDrive"
SHARED_DRIVE = DRIVE_ROOT / "Shareddrives"

print("MyDrive top-level (first 40):")
if MY_DRIVE.exists():
    for item in sorted(MY_DRIVE.iterdir())[:40]:
        print(f"  {item.name}{'/' if item.is_dir() else ''}")

print("\nShared Drives:")
if SHARED_DRIVE.exists():
    for item in sorted(SHARED_DRIVE.iterdir()):
        print(f"  {item.name}/")

TRAINING_ZIP_DIR = None
for c in [MY_DRIVE / FOLDER_NAME, SHARED_DRIVE / FOLDER_NAME]:
    try:
        if c.exists() and c.is_dir():
            zc = len(list(c.glob("*.zip")))
            print(f"\n[FOUND] {c}  ({zc} zips)")
            if zc > 0 and TRAINING_ZIP_DIR is None:
                TRAINING_ZIP_DIR = c
    except PermissionError:
        print(f"  [PERMISSION ERROR] {c}")

if TRAINING_ZIP_DIR:
    print(f"\n[OK] TRAINING_ZIP_DIR = {TRAINING_ZIP_DIR}")
else:
    print("\n[WARN] Auto-detection failed. Set manually in Cell 2.1.")

MyDrive top-level (first 40):
  0_Introduction.gdoc
  2 Pdf_12_09_11_24_09.pdf
  2017831029.gdoc
  2017831036.gdoc
  2017831036_Verse.pptx
  2017831044_B.gdoc
  2017831046/
  2017831046 (1).pdf
  2017831046 (2).pdf
  2017831046 (3).pdf
  2017831046 (4).pdf
  2017831046 (5).pdf
  2017831046 (6).pdf
  2017831046 (7).pdf
  2017831046 (8).pdf
  2017831046.gdoc
  2017831046.pdf
  2017831046_A.pdf
  2017831046_B.pdf
  2017831046_Internship Report PSL.pdf
  20180914_115455.jpg
  20180914_115631.jpg
  20180914_115745.jpg
  20180914_115749.jpg
  20180914_132000.jpg
  20180914_132003.jpg
  20180914_132103.jpg
  20180914_132111.jpg
  20180914_133643.jpg
  20180914_145911.jpg
  20180914_151243.jpg
  20180914_151413 (1).jpg
  20180914_151413.jpg
  20181011_125902.jpg
  20190129_220734.jpg
  20190314_160847.jpg
  20190314_160849.jpg
  20190314_160852.jpg
  20190314_173758.jpg
  20190314_174943.jpg

Shared Drives:

[FOUND] /content/drive/MyDrive/ITS_Intersection_USDOT  (4 zips)

[OK] TRAINING_ZIP_DIR

### 2.1 · Set Paths

In [5]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("/content/intersection_safety_trigger_project")

# ── Manual override if auto-detection failed ──────────────────────────────────
# TRAINING_ZIP_DIR = Path("/content/drive/MyDrive/ITS_Intersection_USDOT")

if "TRAINING_ZIP_DIR" not in dir() or TRAINING_ZIP_DIR is None:
    raise RuntimeError("TRAINING_ZIP_DIR not set. Uncomment the override above.")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
TABLES_DIR  = OUTPUTS_DIR  / "tables"
FIGURES_DIR = OUTPUTS_DIR  / "figures"
SAMPLES_DIR = OUTPUTS_DIR  / "samples"
LOGS_DIR    = OUTPUTS_DIR  / "logs"

print(f"PROJECT_ROOT     = {PROJECT_ROOT}")
print(f"SRC_DIR exists   = {SRC_DIR.exists()}")
print(f"TRAINING_ZIP_DIR = {TRAINING_ZIP_DIR}")

PROJECT_ROOT     = /content/intersection_safety_trigger_project
SRC_DIR exists   = True
TRAINING_ZIP_DIR = /content/drive/MyDrive/ITS_Intersection_USDOT


### 2.2 · Parameters

In [6]:
MAX_RUNS_TO_INSPECT             = 3   # radar+trigger runs to sample deeply
MAX_RECORDS_TO_PRINT            = 3   # MQTT records to print per trigger file
MAX_EVENTS_PER_RECORD_TO_PRINT  = 3   # trigger events to print per record
MAX_TRIGGER_TIMESTAMPS_TO_ALIGN = 50  # trigger timestamps for alignment probe
MAX_RADAR_RECORDS_FOR_CROSSCHECK = 20 # radar records to read for lane/zone check

print(f"MAX_RUNS_TO_INSPECT             = {MAX_RUNS_TO_INSPECT}")
print(f"MAX_RECORDS_TO_PRINT            = {MAX_RECORDS_TO_PRINT}")
print(f"MAX_TRIGGER_TIMESTAMPS_TO_ALIGN = {MAX_TRIGGER_TIMESTAMPS_TO_ALIGN}")
print(f"MAX_RADAR_RECORDS_FOR_CROSSCHECK = {MAX_RADAR_RECORDS_FOR_CROSSCHECK}")

MAX_RUNS_TO_INSPECT             = 3
MAX_RECORDS_TO_PRINT            = 3
MAX_TRIGGER_TIMESTAMPS_TO_ALIGN = 50
MAX_RADAR_RECORDS_FOR_CROSSCHECK = 20


### 2.3 · Imports

In [7]:
import json, re, zipfile, datetime
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd

from file_discovery import find_training_zips, make_output_dirs
from utils import format_size, yes_no, check_mark
from radar_parser import (
    safe_json_load_from_zip,
    decode_payload_if_needed,
    find_timestamp_fields,
    extract_object_list,
    get_object_list_field_name,
    guess_timestamp_format,
    OBJECT_LIST_FIELDS,
)
from trigger_parser import (
    find_trigger_files_in_zip,
    find_radar_path_for_run,
    find_field_paths_recursive,
    extract_trigger_events,
    normalize_trigger_record,
    parse_topic_string,
    iso_to_epoch_ms,
    TRIGGER_SEMANTIC_FIELDS,
    TRIGGER_LIST_FIELDS,
)

print("[OK] All imports successful.")

[OK] All imports successful.


### 2.4 · Create Output Directories & Locate Zip Files

In [8]:
make_output_dirs(TABLES_DIR, FIGURES_DIR, SAMPLES_DIR, LOGS_DIR)

zip_paths = find_training_zips(TRAINING_ZIP_DIR)
if not zip_paths:
    raise FileNotFoundError(f"No zips found in {TRAINING_ZIP_DIR}")

zip_path_map = {zp.name: zp for zp in zip_paths}
print("Zip files:")
for zp in zip_paths:
    print(f"  {zp.name}  ({format_size(zp.stat().st_size)})")

[INFO]  Output directory ready: /content/intersection_safety_trigger_project/outputs/tables
[INFO]  Output directory ready: /content/intersection_safety_trigger_project/outputs/figures
[INFO]  Output directory ready: /content/intersection_safety_trigger_project/outputs/samples
[INFO]  Output directory ready: /content/intersection_safety_trigger_project/outputs/logs
[INFO]  Found 4 zip file(s) in: /content/drive/MyDrive/ITS_Intersection_USDOT
Zip files:
  Training Data 1.zip  (151.06 GB)
  Training Data 2.zip  (150.50 GB)
  Training Data 3.zip  (152.57 GB)
  Training Data 4.zip  (149.71 GB)


---
## 3 · Discover Trigger Files Across All Zips

In [9]:
print("Scanning for trigger JSON files (central directory only) ...\n")

all_trigger_files: list[dict] = []
for zip_path in zip_paths:
    print(f"  {zip_path.name} ...", end=" ", flush=True)
    files = find_trigger_files_in_zip(zip_path)
    all_trigger_files.extend(files)
    print(f"{len(files)} trigger files")

trig_inv_df = pd.DataFrame(all_trigger_files)

print(f"\n[OK] Total trigger files : {len(all_trigger_files):,}")
print(f"     Unique runs          : {trig_inv_df['run_id'].nunique():,}")
print(f"     File sizes (KB) median: {trig_inv_df['file_size_kb'].median():.1f}")

out_path = TABLES_DIR / "trigger_file_inventory.csv"
trig_inv_df.to_csv(out_path, index=False)
print(f"\n[SAVED] {out_path}")
trig_inv_df.head(6)

Scanning for trigger JSON files (central directory only) ...

  Training Data 1.zip ... 104 trigger files
  Training Data 2.zip ... 99 trigger files
  Training Data 3.zip ... 100 trigger files
  Training Data 4.zip ... 88 trigger files

[OK] Total trigger files : 391
     Unique runs          : 391
     File sizes (KB) median: 6380.6

[SAVED] /content/intersection_safety_trigger_project/outputs/tables/trigger_file_inventory.csv


,zip_name,internal_path,run_id,file_size_kb
0,Training Data 1.zip,Run_166/Radars_Run_166_traffic-triggers-output...,0166,7054.7
1,Training Data 1.zip,Run_844/Radars_Run_844_traffic-triggers-output...,0844,4140.1
2,Training Data 1.zip,Run_327/Radars_Run_327_traffic-triggers-output...,0327,6282.3
3,Training Data 1.zip,Run_717/Radars_Run_717_traffic-triggers-output...,0717,6804.2
4,Training Data 1.zip,Run_257/Radars_Run_257_traffic-triggers-output...,0257,2544.6
5,Training Data 1.zip,Run_1016/Radars_Run_1016_traffic-triggers-outp...,1016,9729.6


---
## 4 · Select Sample Runs

Select runs that have **both** a trigger file **and** all 4 radar sensor files,
from different zips where possible.

In [10]:
from radar_parser import find_radar_files_in_zip

# Build set of runs that have all 4 radar sensors
print("Scanning for radar files to find runs with full sensor coverage ...")
radar_run_set: dict[tuple, list] = defaultdict(list)   # (zip_name, run_id) -> sensor_ids

for zip_path in zip_paths:
    print(f"  {zip_path.name} ...", end=" ", flush=True)
    radar_files = find_radar_files_in_zip(zip_path)
    for rf in radar_files:
        key = (rf["zip_name"], rf["run_id"])
        radar_run_set[key].append(rf["sensor_id"])
    print(f"{len(radar_files)} radar files")

full_radar_runs = {k for k, sensors in radar_run_set.items() if len(sensors) >= 4}

# Find runs that have BOTH trigger and full radar coverage
trigger_run_set = set(
    (row["zip_name"], row["run_id"])
    for _, row in trig_inv_df.iterrows()
)
candidate_runs = sorted(full_radar_runs & trigger_run_set)
print(f"\nRuns with trigger + all 4 radar sensors: {len(candidate_runs)}")

# Pick MAX_RUNS_TO_INSPECT preferring different zips
selected: list[tuple] = []
seen_zips: set = set()
for zn, rid in candidate_runs:
    if len(selected) >= MAX_RUNS_TO_INSPECT:
        break
    if zn not in seen_zips:
        selected.append((zn, rid))
        seen_zips.add(zn)
for zn, rid in candidate_runs:
    if len(selected) >= MAX_RUNS_TO_INSPECT:
        break
    if (zn, rid) not in selected:
        selected.append((zn, rid))

print(f"\nSelected {len(selected)} run(s):")
for zn, rid in selected:
    sensors = sorted(radar_run_set[(zn, rid)])
    print(f"  Run {rid}  |  {zn}  |  sensors: {sensors}")

Scanning for radar files to find runs with full sensor coverage ...
  Training Data 1.zip ... 416 radar files
  Training Data 2.zip ... 396 radar files
  Training Data 3.zip ... 400 radar files
  Training Data 4.zip ... 352 radar files

Runs with trigger + all 4 radar sensors: 391

Selected 3 run(s):
  Run 0048  |  Training Data 1.zip  |  sensors: [1, 2, 3, 4]
  Run 0045  |  Training Data 2.zip  |  sensors: [1, 2, 3, 4]
  Run 0001  |  Training Data 3.zip  |  sensors: [1, 2, 3, 4]


---
## 5 · Read and Inspect Trigger JSON Files

For each sampled run, open the trigger file directly from the zip
and inspect its structure.

In [11]:
# Central accumulator: (zip_name, run_id) -> inspection result dict
trig_results: dict[tuple, dict] = {}

for zip_name, run_id in selected:
    zip_path = zip_path_map[zip_name]

    # Get internal path from inventory
    mask = (trig_inv_df["zip_name"] == zip_name) & (trig_inv_df["run_id"] == run_id)
    rows = trig_inv_df[mask]
    if rows.empty:
        print(f"[SKIP] Run {run_id} / {zip_name} — no trigger file in inventory")
        continue
    internal_path = rows.iloc[0]["internal_path"]
    file_size_kb  = rows.iloc[0]["file_size_kb"]

    print(f"\n{'='*66}")
    print(f"  Run {run_id}  |  {zip_name}  ({file_size_kb:.1f} KB)")
    print(f"  {internal_path}")
    print(f"{'='*66}")

    result = {
        "zip_name":          zip_name,
        "run_id":            run_id,
        "internal_path":     internal_path,
        "file_size_kb":      file_size_kb,
        "top_level_type":    None,
        "num_records":       0,
        "top_level_keys":    [],
        "payload_encoding":  None,
        "payload_type":      None,
        "payload_keys":      [],
        "topic_samples":     [],
        "timestamp_fields":  {},
        "all_nested_paths":  {},    # field_name -> [(path, val)]
        "trigger_events":    [],    # normalized event rows
        "num_records_with_events": 0,
        "num_records_empty":       0,
        "error":             None,
    }

    data = safe_json_load_from_zip(zip_path, internal_path)
    if data is None:
        result["error"] = "failed to load"
        trig_results[(zip_name, run_id)] = result
        continue

    if isinstance(data, list):
        result["top_level_type"] = "list"
        result["num_records"]    = len(data)
        print(f"  top-level: list  ({len(data):,} records)")

        sample_records = [r for r in data[:MAX_RECORDS_TO_PRINT] if isinstance(r, dict)]
        if sample_records:
            result["top_level_keys"] = list(sample_records[0].keys())
            print(f"  record keys: {result['top_level_keys']}")

        # Process ALL records (not just sample) for stats
        for record in data:
            if not isinstance(record, dict):
                continue
            # Collect topic samples
            topic = record.get("topic", "")
            if topic and topic not in result["topic_samples"]:
                result["topic_samples"].append(topic)

            # Collect timestamps
            raw_ts = record.get("receivedAt")
            if raw_ts and "receivedAt" not in result["timestamp_fields"]:
                result["timestamp_fields"]["receivedAt"] = str(raw_ts)

            # Decode payload
            raw_payload = record.get("payload")
            if raw_payload is None:
                result["num_records_empty"] += 1
                continue
            payload, encoding = decode_payload_if_needed(raw_payload)
            result["payload_encoding"] = encoding
            if isinstance(payload, dict):
                result["payload_type"] = "dict"
                if not result["payload_keys"]:
                    result["payload_keys"] = list(payload.keys())
                # Payload timestamp
                for ts_k in ("timestamp", "time", "capture_time", "event_time"):
                    if payload.get(ts_k) and f"payload.{ts_k}" not in result["timestamp_fields"]:
                        result["timestamp_fields"][f"payload.{ts_k}"] = str(payload[ts_k])

                # Nested field search (on all records)
                paths = find_field_paths_recursive(payload, TRIGGER_SEMANTIC_FIELDS)
                for field, hits in paths.items():
                    if field not in result["all_nested_paths"]:
                        result["all_nested_paths"][field] = hits[:3]

                # Extract trigger events
                events = normalize_trigger_record(record, run_id, zip_name)
                if events:
                    result["trigger_events"].extend(events)
                    result["num_records_with_events"] += 1
                else:
                    result["num_records_empty"] += 1

        # Print summary for sampled records
        for rec_idx, record in enumerate(sample_records):
            print(f"\n  record[{rec_idx}]:")
            print(f"    receivedAt : {record.get('receivedAt', '(none)')}")
            print(f"    topic      : {record.get('topic', '(none)')}")
            raw_p = record.get("payload")
            payload, enc = decode_payload_if_needed(raw_p) if raw_p is not None else (None, "none")
            print(f"    payload enc: {enc}")
            if isinstance(payload, dict):
                print(f"    payload keys: {list(payload.keys())}")
                events = extract_trigger_events(payload)
                print(f"    trigger events found: {len(events)}")
                for ev_i, ev in enumerate(events[:MAX_EVENTS_PER_RECORD_TO_PRINT]):
                    print(f"      event[{ev_i}]: ref={ev.get('reference_name')}  "
                          f"lane={ev.get('associated_lane')}  "
                          f"zone={ev.get('associated_zone')}  "
                          f"sensor={ev.get('associated_sensor')}")

    elif isinstance(data, dict):
        result["top_level_type"] = "dict"
        result["top_level_keys"] = list(data.keys())
        print(f"  top-level: dict  keys={result['top_level_keys']}")
        paths = find_field_paths_recursive(data, TRIGGER_SEMANTIC_FIELDS)
        result["all_nested_paths"] = {k: v[:3] for k, v in paths.items()}

    print(f"\n  Records with events : {result['num_records_with_events']:,}")
    print(f"  Records empty       : {result['num_records_empty']:,}")
    print(f"  Total events found  : {len(result['trigger_events']):,}")
    trig_results[(zip_name, run_id)] = result

print(f"\n[OK] Trigger inspection complete — {len(trig_results)} run(s).")


  Run 0048  |  Training Data 1.zip  (6856.3 KB)
  Run_48/Radars_Run_48_traffic-triggers-output.json
  top-level: list  (1,965 records)
  record keys: ['topic', 'payload', 'qos', 'receivedAt', 'retain']

  record[0]:
    receivedAt : 2024-01-31 11:27:51
    topic      : traffic-triggers-output
    payload enc: dict_or_list
    payload keys: ['timestamp', 'trigger_outputs']
    trigger events found: 1
      event[0]: ref=None  lane=None  zone=None  sensor=None

  record[1]:
    receivedAt : 2024-01-31 11:27:51
    topic      : traffic-triggers-output
    payload enc: dict_or_list
    payload keys: ['timestamp', 'trigger_outputs']
    trigger events found: 1
      event[0]: ref=None  lane=None  zone=None  sensor=None

  record[2]:
    receivedAt : 2024-01-31 11:27:51
    topic      : traffic-triggers-output
    payload enc: dict_or_list
    payload keys: ['timestamp', 'trigger_outputs']
    trigger events found: 1
      event[0]: ref=None  lane=None  zone=None  sensor=None

  Records wit

---
## 6 · Nested Field Paths

Where exactly do the semantic fields live inside the payload?

In [12]:
path_rows = []
for (zip_name, run_id), result in trig_results.items():
    if result.get("error"):
        continue
    for field_name, hits in result.get("all_nested_paths", {}).items():
        for path, val in hits[:2]:  # save up to 2 occurrences per field per run
            path_rows.append({
                "zip_name":    zip_name,
                "run_id":      run_id,
                "field_name":  field_name,
                "dotted_path": path,
                "value_type":  type(val).__name__,
                "sample_value": str(val)[:120],
            })

if path_rows:
    paths_df = pd.DataFrame(path_rows)
    out_path = TABLES_DIR / "trigger_nested_field_paths.csv"
    paths_df.to_csv(out_path, index=False)
    print(f"[SAVED] {out_path}")

    print("\nSemantic fields discovered (unique paths):")
    print(f"{'Field':<25s} {'Path':<55s} {'Type':<10s} {'Sample value'}")
    print("-" * 120)
    for _, row in paths_df.drop_duplicates(["field_name", "dotted_path"]).iterrows():
        print(f"  {row['field_name']:<23s} {row['dotted_path']:<53s} "
              f"{row['value_type']:<10s} {row['sample_value'][:35]}")
else:
    paths_df = pd.DataFrame()
    print("[WARN] No semantic fields found in payload.")
    print("       Check the record printout in Section 5 to see what payload contains.")

[SAVED] /content/intersection_safety_trigger_project/outputs/tables/trigger_nested_field_paths.csv

Semantic fields discovered (unique paths):
Field                     Path                                                    Type       Sample value
------------------------------------------------------------------------------------------------------------------------
  trigger_outputs         trigger_outputs                                       list       [{'traffic_triggers': [{'associated
  traffic_triggers        trigger_outputs[0].traffic_triggers                   list       [{'associated_lane': 'lane22', 'ass
  associated_lane         trigger_outputs[0].traffic_triggers[0].associated_lane str        lane22
  associated_lane         trigger_outputs[0].traffic_triggers[1].associated_lane str        lane24
  associated_sensor       trigger_outputs[0].traffic_triggers[0].associated_sensor str        sensor3
  associated_sensor       trigger_outputs[0].traffic_triggers[1].associated_

---
## 7 · Sample Trigger Event Table

One row per trigger event extracted from the sampled runs.

In [13]:
all_events: list[dict] = []
for (zip_name, run_id), result in trig_results.items():
    all_events.extend(result.get("trigger_events", []))

if all_events:
    events_df = pd.DataFrame(all_events)
    out_path  = TABLES_DIR / "sample_trigger_event_table.csv"
    events_df.to_csv(out_path, index=False)
    print(f"[SAVED] {out_path}  (shape: {events_df.shape})")
    print(f"\nTotal events      : {len(events_df)}")
    print(f"Unique ref names  : {events_df['reference_name'].nunique()}")
    print(f"Unique lanes      : {events_df['associated_lane'].nunique()}")
    print(f"Unique zones      : {events_df['associated_zone'].nunique()}")
    print(f"Unique sensors    : {events_df['associated_sensor'].nunique()}")
    print(f"\nref_name value counts (top 15):")
    print(events_df["reference_name"].value_counts().head(15).to_string())
    display(events_df.head(10))
else:
    events_df = pd.DataFrame()
    print("[WARN] No trigger events were extracted.")
    print("       Trigger events may be stored under a different field name.")
    print("       Check the nested field paths in Section 6 for clues.")

[SAVED] /content/intersection_safety_trigger_project/outputs/tables/sample_trigger_event_table.csv  (shape: (4332, 12))

Total events      : 4332
Unique ref names  : 0
Unique lanes      : 0
Unique zones      : 0
Unique sensors    : 0

ref_name value counts (top 15):
Series([], )


,run_id,zip_name,receivedAt,payload_timestamp,topic,reference_name,associated_lane,associated_zone,associated_sensor,trigger_output_index,trigger_index,container_field
0,0048,Training Data 1.zip,2024-01-31 11:27:51,2024-01-31T16:28:38+00:00,traffic-triggers-output,None,None,None,None,0,0,trigger_outputs
1,0048,Training Data 1.zip,2024-01-31 11:27:51,2024-01-31T16:28:38+00:00,traffic-triggers-output,None,None,None,None,0,0,trigger_outputs
2,0048,Training Data 1.zip,2024-01-31 11:27:51,2024-01-31T16:28:38+00:00,traffic-triggers-output,None,None,None,None,0,0,trigger_outputs
3,0048,Training Data 1.zip,2024-01-31 11:27:51,2024-01-31T16:28:38+00:00,traffic-triggers-output,None,None,None,None,0,0,trigger_outputs
4,0048,Training Data 1.zip,2024-01-31 11:27:51,2024-01-31T16:28:38+00:00,traffic-triggers-output,None,None,None,None,0,0,trigger_outputs
5,0048,Training Data 1.zip,2024-01-31 11:27:51,2024-01-31T16:28:38+00:00,traffic-triggers-output,None,None,None,None,0,0,trigger_outputs
6,0048,Training Data 1.zip,2024-01-31 11:27:51,2024-01-31T16:28:39+00:00,traffic-triggers-output,None,None,None,None,0,0,trigger_outputs
7,0048,Training Data 1.zip,2024-01-31 11:27:51,2024-01-31T16:28:39+00:00,traffic-triggers-output,None,None,None,None,0,0,trigger_outputs
8,0048,Training Data 1.zip,2024-01-31 11:27:51,2024-01-31T16:28:39+00:00,traffic-triggers-output,None,None,None,None,0,0,trigger_outputs
9,0048,Training Data 1.zip,2024-01-31 11:27:51,2024-01-31T16:28:39+00:00,traffic-triggers-output,None,None,None,None,0,0,trigger_outputs


---
## 8 · Trigger Reference Mapping Table

`reference_name → lane / zone / sensor` with occurrence counts.

> Treat `reference_name` as a detector/zone activation identifier,
> not a confirmed semantic event category.

In [14]:
if not events_df.empty and "reference_name" in events_df.columns:
    # Group by reference_name + lane + zone + sensor
    group_cols = ["reference_name", "associated_lane",
                  "associated_zone", "associated_sensor"]
    mapping_df = (
        events_df
        .groupby(group_cols, dropna=False)
        .agg(
            count       = ("run_id", "count"),
            example_run = ("run_id", "first"),
            example_zip = ("zip_name", "first"),
            example_topic = ("topic", "first"),
        )
        .reset_index()
        .sort_values("count", ascending=False)
    )
    out_path = TABLES_DIR / "sample_trigger_reference_mapping.csv"
    mapping_df.to_csv(out_path, index=False)
    print(f"[SAVED] {out_path}  (shape: {mapping_df.shape})")
    print()
    display(mapping_df.head(20))
else:
    mapping_df = pd.DataFrame()
    print("[WARN] No events to build mapping table from.")

[SAVED] /content/intersection_safety_trigger_project/outputs/tables/sample_trigger_reference_mapping.csv  (shape: (1, 8))



,reference_name,associated_lane,associated_zone,associated_sensor,count,example_run,example_zip,example_topic
0,NaN,NaN,NaN,NaN,4332,0048,Training Data 1.zip,traffic-triggers-output


---
## 9 · MQTT Topic Inspection

Parse the topic strings to look for embedded lane/zone/sensor/intersection info.

In [15]:
# Collect all unique topic strings from inspected runs
all_topics: set[str] = set()
for result in trig_results.values():
    for t in result.get("topic_samples", []):
        if t:
            all_topics.add(t)

topic_rows = [parse_topic_string(t) for t in sorted(all_topics)]

if topic_rows:
    topics_df = pd.DataFrame(topic_rows).drop(columns=["parts"])
    out_path  = TABLES_DIR / "trigger_topic_samples.csv"
    topics_df.to_csv(out_path, index=False)
    print(f"[SAVED] {out_path}")
    print(f"\nUnique topics: {len(topic_rows)}")
    print()
    print(f"{'Topic string':<70s}  sensor  lane  zone  intersection")
    print("-" * 120)
    for row in topic_rows[:25]:
        print(f"  {row['raw_topic']:<68s}  "
              f"{str(row['likely_sensor']):<8s}"
              f"{str(row['likely_lane']):<8s}"
              f"{str(row['likely_zone']):<8s}"
              f"{str(row['likely_intersection'])}")
    if len(topic_rows) > 25:
        print(f"  ... and {len(topic_rows) - 25} more")
else:
    topics_df = pd.DataFrame()
    print("[WARN] No MQTT topic strings found.")

[SAVED] /content/intersection_safety_trigger_project/outputs/tables/trigger_topic_samples.csv

Unique topics: 1

Topic string                                                            sensor  lane  zone  intersection
------------------------------------------------------------------------------------------------------------------------
  traffic-triggers-output                                               None    None    None    None


---
## 10 · Trigger Frequency Summary

Per-run counts: total records, records with events, empty records,
unique reference names, lanes, zones, and sensors.

In [16]:
freq_rows = []
for (zip_name, run_id), result in trig_results.items():
    if result.get("error"):
        continue
    run_events = pd.DataFrame(result.get("trigger_events", []))
    freq_rows.append({
        "zip_name":             zip_name,
        "run_id":               run_id,
        "total_records":        result.get("num_records", 0),
        "records_with_events":  result.get("num_records_with_events", 0),
        "records_empty":        result.get("num_records_empty", 0),
        "total_events":         len(run_events),
        "unique_reference_names": run_events["reference_name"].nunique() if not run_events.empty else 0,
        "unique_lanes":         run_events["associated_lane"].nunique()  if not run_events.empty else 0,
        "unique_zones":         run_events["associated_zone"].nunique()  if not run_events.empty else 0,
        "unique_sensors":       run_events["associated_sensor"].nunique()if not run_events.empty else 0,
        "payload_encoding":     result.get("payload_encoding", ""),
        "payload_keys":         str(result.get("payload_keys", [])),
    })

freq_df = pd.DataFrame(freq_rows)
out_path = TABLES_DIR / "sample_trigger_frequency_summary.csv"
freq_df.to_csv(out_path, index=False)
print(f"[SAVED] {out_path}")
display(freq_df)

[SAVED] /content/intersection_safety_trigger_project/outputs/tables/sample_trigger_frequency_summary.csv


,zip_name,run_id,total_records,records_with_events,records_empty,total_events,unique_reference_names,unique_lanes,unique_zones,unique_sensors,payload_encoding,payload_keys
0,Training Data 1.zip,0048,1965,1801,164,1801,0,0,0,0,dict_or_list,"['timestamp', 'trigger_outputs']"
1,Training Data 2.zip,0045,1808,798,1010,798,0,0,0,0,dict_or_list,"['timestamp', 'trigger_outputs']"
2,Training Data 3.zip,0001,3236,1733,1503,1733,0,0,0,0,dict_or_list,"['timestamp', 'trigger_outputs']"


---
## 11 · Radar–Trigger Lane/Zone Cross-Check

Compare the lane/zone identifiers found in radar object fields
(`closest_lane`, `within_zone`) with those found in trigger events
(`associated_lane`, `associated_zone`).

In [17]:
crosscheck_rows = []

for zip_name, run_id in selected:
    zip_path = zip_path_map[zip_name]
    result   = trig_results.get((zip_name, run_id), {})

    # ── Trigger lane/zone values ─────────────────────────────────────────────
    run_events = pd.DataFrame(result.get("trigger_events", []))
    trig_lanes = set() if run_events.empty else set(
        run_events["associated_lane"].dropna().unique()
    )
    trig_zones = set() if run_events.empty else set(
        run_events["associated_zone"].dropna().unique()
    )

    # ── Radar lane/zone values (read sensor1) ────────────────────────────────
    radar_lanes: set = set()
    radar_zones: set = set()

    radar_path = find_radar_path_for_run(zip_path, run_id, sensor_id=1)
    if radar_path:
        radar_data = safe_json_load_from_zip(zip_path, radar_path)
        if isinstance(radar_data, list):
            records_checked = 0
            for record in radar_data:
                if records_checked >= MAX_RADAR_RECORDS_FOR_CROSSCHECK:
                    break
                if not isinstance(record, dict):
                    continue
                raw_p = record.get("payload")
                if raw_p is None:
                    continue
                payload, _ = decode_payload_if_needed(raw_p)
                if not isinstance(payload, dict):
                    continue
                obj_list = extract_object_list(payload)
                if not obj_list:
                    continue
                for obj in obj_list:
                    if not isinstance(obj, dict):
                        continue
                    for lf in ("closest_lane", "lane", "lane_id"):
                        if obj.get(lf) is not None:
                            radar_lanes.add(str(obj[lf]))
                    for zf in ("within_zone", "zone", "zone_id"):
                        if obj.get(zf) is not None:
                            radar_zones.add(str(obj[zf]))
                records_checked += 1

    # ── Compare ──────────────────────────────────────────────────────────────
    lane_overlap = radar_lanes & trig_lanes
    zone_overlap = radar_zones & trig_zones

    crosscheck_rows.append({
        "zip_name":         zip_name,
        "run_id":           run_id,
        "trigger_lanes":    "|".join(sorted(trig_lanes)),
        "trigger_zones":    "|".join(sorted(trig_zones)),
        "radar_lanes":      "|".join(sorted(radar_lanes)),
        "radar_zones":      "|".join(sorted(radar_zones)),
        "lane_overlap":     "|".join(sorted(lane_overlap)),
        "zone_overlap":     "|".join(sorted(zone_overlap)),
        "lanes_compatible": len(lane_overlap) > 0 or (not trig_lanes or not radar_lanes),
        "zones_compatible": len(zone_overlap) > 0 or (not trig_zones or not radar_zones),
        "note":             "overlap found" if (lane_overlap or zone_overlap)
                            else ("no overlap" if (trig_lanes and radar_lanes) else "missing data"),
    })

    print(f"\n  Run {run_id} | {zip_name}")
    print(f"    Trigger lanes  : {sorted(trig_lanes) or '(none found)'}")
    print(f"    Radar lanes    : {sorted(radar_lanes) or '(none found)'}")
    print(f"    Lane overlap   : {sorted(lane_overlap) or '(none)'}")
    print(f"    Trigger zones  : {sorted(trig_zones) or '(none found)'}")
    print(f"    Radar zones    : {sorted(radar_zones) or '(none found)'}")
    print(f"    Zone overlap   : {sorted(zone_overlap) or '(none)'}")

crosscheck_df = pd.DataFrame(crosscheck_rows)
out_path = TABLES_DIR / "sample_radar_trigger_lane_zone_comparison.csv"
crosscheck_df.to_csv(out_path, index=False)
print(f"\n[SAVED] {out_path}")


  Run 0048 | Training Data 1.zip
    Trigger lanes  : (none found)
    Radar lanes    : ['lane10']
    Lane overlap   : (none)
    Trigger zones  : (none found)
    Radar zones    : ["['zoneAI']"]
    Zone overlap   : (none)

  Run 0045 | Training Data 2.zip
    Trigger lanes  : (none found)
    Radar lanes    : ['lane10']
    Lane overlap   : (none)
    Trigger zones  : (none found)
    Radar zones    : ['[]']
    Zone overlap   : (none)

  Run 0001 | Training Data 3.zip
    Trigger lanes  : (none found)
    Radar lanes    : ['lane10']
    Lane overlap   : (none)
    Trigger zones  : (none found)
    Radar zones    : ["['zoneAI']"]
    Zone overlap   : (none)

[SAVED] /content/intersection_safety_trigger_project/outputs/tables/sample_radar_trigger_lane_zone_comparison.csv


---
## 12 · Timestamp Alignment Probe

For each sampled run, compare trigger `receivedAt` timestamps with the
nearest radar `receivedAt` timestamps and compute the gap in milliseconds.

In [18]:
alignment_rows = []

for zip_name, run_id in selected:
    zip_path = zip_path_map[zip_name]
    result   = trig_results.get((zip_name, run_id), {})
    if result.get("error"):
        continue

    # ── Collect trigger receivedAt timestamps ────────────────────────────────
    trig_data_path = trig_inv_df.loc[
        (trig_inv_df["zip_name"] == zip_name) &
        (trig_inv_df["run_id"] == run_id), "internal_path"
    ].iloc[0]
    trig_data = safe_json_load_from_zip(zip_path, trig_data_path)
    trig_ts_ms: list[float] = []
    if isinstance(trig_data, list):
        for rec in trig_data:
            if len(trig_ts_ms) >= MAX_TRIGGER_TIMESTAMPS_TO_ALIGN:
                break
            ms = iso_to_epoch_ms(rec.get("receivedAt"))
            if ms is not None:
                trig_ts_ms.append(ms)

    # ── Collect radar receivedAt timestamps from sensor1 ─────────────────────
    radar_path = find_radar_path_for_run(zip_path, run_id, sensor_id=1)
    radar_ts_ms: list[float] = []
    if radar_path:
        radar_data = safe_json_load_from_zip(zip_path, radar_path)
        if isinstance(radar_data, list):
            for rec in radar_data:
                ms = iso_to_epoch_ms(rec.get("receivedAt"))
                if ms is not None:
                    radar_ts_ms.append(ms)

    # ── Compute nearest gap for each trigger timestamp ────────────────────────
    if not radar_ts_ms:
        print(f"  [SKIP] Run {run_id} — no radar timestamps found")
        continue

    radar_arr = sorted(radar_ts_ms)
    n_aligned  = 0
    gaps: list[float] = []

    for t_ms in trig_ts_ms:
        # Binary-search-style nearest: find smallest |t_ms - r|
        best_gap = min(abs(t_ms - r) for r in radar_arr)
        gaps.append(best_gap)
        alignment_rows.append({
            "zip_name":             zip_name,
            "run_id":               run_id,
            "trigger_ts_ms":        t_ms,
            "nearest_radar_gap_ms": best_gap,
        })
        n_aligned += 1

    if gaps:
        print(f"  Run {run_id} | trigger stamps: {len(trig_ts_ms)} | radar stamps: {len(radar_ts_ms)}")
        print(f"    Gap (ms): min={min(gaps):.1f}  median={sorted(gaps)[len(gaps)//2]:.1f}  "
              f"max={max(gaps):.1f}")
        sub_100ms = sum(1 for g in gaps if g < 100)
        print(f"    Within 100 ms: {sub_100ms}/{len(gaps)} = "
              f"{100*sub_100ms/len(gaps):.0f}%")

if alignment_rows:
    align_df = pd.DataFrame(alignment_rows)
    out_path  = TABLES_DIR / "sample_trigger_radar_alignment_probe.csv"
    align_df.to_csv(out_path, index=False)
    print(f"\n[SAVED] {out_path}  (shape: {align_df.shape})")
    print(f"\nOverall gap stats (ms):")
    print(align_df["nearest_radar_gap_ms"].describe().round(1).to_string())
else:
    align_df = pd.DataFrame()
    print("[WARN] No alignment data — check timestamp fields.")

  [SKIP] Run 0048 — no radar timestamps found
  [SKIP] Run 0045 — no radar timestamps found
  [SKIP] Run 0001 — no radar timestamps found
[WARN] No alignment data — check timestamp fields.


---
## 13 · Auto-Generated Findings Summary

In [19]:
import datetime

L = []
def ln(s=""): L.append(s)

good = [r for r in trig_results.values() if not r.get("error")]
top_types = Counter(r["top_level_type"] for r in good if r["top_level_type"])
pay_encs  = Counter(r["payload_encoding"] for r in good if r["payload_encoding"])

# Container field found
container_found = None
if not events_df.empty and "container_field" in events_df.columns:
    cf = events_df["container_field"].value_counts()
    container_found = cf.index[0] if len(cf) else None

ln("# Notebook 02 — Trigger Log Deep Inspection Findings")
ln("**Physics-Informed Trigger Event Analysis**  ")
ln(f"**Generated:** {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}  ")
ln(f"**Runs inspected:** {len(trig_results)} | **Total events extracted:** {len(events_df)}")
ln()
ln("> **Semantic warning:** `reference_name` values are treated as detector/zone")
ln("> activation identifiers until confirmed by cross-modal evidence.")
ln()
ln("---")

ln("\n## 1. Trigger JSON Top-Level Structure")
ln(f"Top-level type: {dict(top_types)}")
if top_types.get("list", 0):
    sample_keys = good[0].get("top_level_keys", []) if good else []
    ln(f"Same MQTT message-log format as radar JSON: `{sample_keys}`")

ln("\n## 2. Payload Encoding")
ln(f"Payload encoding: {dict(pay_encs)}")
if pay_encs.get("dict_or_list"):
    ln("`payload` is already a Python dict — no additional decoding required.")
elif pay_encs.get("string_json"):
    ln("`payload` is a JSON-encoded string — must call `json.loads()` before use.")
elif pay_encs.get("base64_json"):
    ln("`payload` is base64-encoded JSON — decode with base64 then json.loads().")
if good:
    ln(f"Payload top-level keys (sample): `{good[0].get('payload_keys', [])}`")

ln("\n## 3. Where Trigger Events Are Stored")
if container_found:
    ln(f"Trigger events found inside: **`payload.{container_found}`**")
    ln(f"Total events across {len(trig_results)} sampled run(s): {len(events_df)}")
elif not events_df.empty:
    ln(f"Trigger events extracted (container field: various). Total: {len(events_df)}")
else:
    ln("No trigger events were extracted in this sample.")
    ln("The payload may use a different container field name not in `TRIGGER_LIST_FIELDS`.")
    ln("Check `trigger_nested_field_paths.csv` for the actual field name.")

ln("\n## 4. Semantic Fields Available")
ln()
ln("| Field | Found | Unique values (sample) |")
ln("|-------|-------|------------------------|")
for col, label in [("reference_name", "reference_name"),
                   ("associated_lane", "associated_lane"),
                   ("associated_zone", "associated_zone"),
                   ("associated_sensor", "associated_sensor")]:
    if not events_df.empty and col in events_df.columns:
        vals = events_df[col].dropna().unique()
        sample_vals = ", ".join(f"`{v}`" for v in vals[:5])
        if len(vals) > 5:
            sample_vals += f" ... ({len(vals)} total)"
        ln(f"| `{label}` | YES ({len(vals)} unique) | {sample_vals} |")
    else:
        ln(f"| `{label}` | NOT FOUND | — |")

ln("\n## 5. MQTT Topic Analysis")
ln(f"Unique topics collected: {len(topics_df) if not topics_df.empty else 0}")
if not topics_df.empty:
    sensor_info = topics_df["likely_sensor"].dropna().unique()
    lane_info   = topics_df["likely_lane"].dropna().unique()
    zone_info   = topics_df["likely_zone"].dropna().unique()
    ln(f"Topics containing sensor info: {len(sensor_info)} — {list(sensor_info)[:5]}")
    ln(f"Topics containing lane info  : {len(lane_info)} — {list(lane_info)[:5]}")
    ln(f"Topics containing zone info  : {len(zone_info)} — {list(zone_info)[:5]}")

ln("\n## 6. Radar–Trigger Lane/Zone Compatibility")
if not crosscheck_df.empty:
    n_lane_compat = crosscheck_df["lanes_compatible"].sum()
    n_zone_compat = crosscheck_df["zones_compatible"].sum()
    ln(f"Runs with lane compatibility: {n_lane_compat}/{len(crosscheck_df)}")
    ln(f"Runs with zone compatibility: {n_zone_compat}/{len(crosscheck_df)}")
    overlaps = crosscheck_df[crosscheck_df["lane_overlap"] != ""]
    if not overlaps.empty:
        ln(f"Overlapping lane values found — radar and trigger use the same identifiers.")
        ln(f"This confirms `closest_lane` can be used to match objects to trigger lanes.")
    else:
        ln("No direct lane overlap found in sample. Possible reasons:")
        ln("- Sampled radar records had no objects.")
        ln("- Lane identifiers use different formats (e.g. int vs string, 0-indexed vs 1-indexed).")
        ln("- Radar records and trigger records for this sample don't time-overlap.")
else:
    ln("Cross-check skipped — no crosscheck data available.")

ln("\n## 7. Timestamp Alignment")
if not align_df.empty:
    gaps = align_df["nearest_radar_gap_ms"]
    sub100 = (gaps < 100).sum()
    ln(f"Trigger timestamps probed: {len(align_df)}")
    ln(f"Gap stats (ms): min={gaps.min():.1f}  median={gaps.median():.1f}  max={gaps.max():.1f}")
    ln(f"Within 100 ms: {sub100}/{len(gaps)} ({100*sub100/len(gaps):.0f}%)")
    if gaps.median() < 500:
        ln("**`receivedAt` is suitable for radar-trigger alignment** — median gap is small.")
    else:
        ln("Median gap is large — alignment may need `payload.timestamp` instead of `receivedAt`.")
else:
    ln("Alignment probe skipped or had no data.")

ln("\n## 8. What Notebook 03 Should Do")
ln("- Load trigger events from all ~400 radar runs (not just 3).")
ln("- Load radar records from the same runs.")
ln("- For each trigger event, define a time window around `receivedAt`.")
ln("- Collect all radar objects from sensor1–sensor4 within that window.")
ln("- Label the window as a trigger activation (positive).")
ln("- Sample an equal number of non-trigger windows (negative).")
ln("- Save the labelled window dataset to `outputs/window_dataset/`.")
ln("- Use `closest_lane` / `within_zone` to filter objects to the trigger's lane/zone.")
ln("")
ln("> Do not start feature engineering or ML until the window dataset is validated.")

summary_text = "\n".join(L)
print(summary_text[:3000], "\n...")

out_path = TABLES_DIR / "notebook_02_trigger_findings.md"
out_path.write_text(summary_text, encoding="utf-8")
print(f"\n[SAVED] {out_path}")

# Notebook 02 — Trigger Log Deep Inspection Findings
**Physics-Informed Trigger Event Analysis**  
**Generated:** 2026-05-14 07:10  
**Runs inspected:** 3 | **Total events extracted:** 4332

> **Semantic warning:** `reference_name` values are treated as detector/zone
> activation identifiers until confirmed by cross-modal evidence.

---

## 1. Trigger JSON Top-Level Structure
Top-level type: {'list': 3}
Same MQTT message-log format as radar JSON: `['topic', 'payload', 'qos', 'receivedAt', 'retain']`

## 2. Payload Encoding
Payload encoding: {'dict_or_list': 3}
`payload` is already a Python dict — no additional decoding required.
Payload top-level keys (sample): `['timestamp', 'trigger_outputs']`

## 3. Where Trigger Events Are Stored
Trigger events found inside: **`payload.trigger_outputs`**
Total events across 3 sampled run(s): 4332

## 4. Semantic Fields Available

| Field | Found | Unique values (sample) |
|-------|-------|------------------------|
| `reference_name` | YES (0 unique

---
## 14 · Package Outputs and Download

In [20]:
import shutil, datetime
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path  = Path(f"/content/task2_outputs_{timestamp}.zip")
shutil.make_archive(str(zip_path.with_suffix("")), "zip", str(OUTPUTS_DIR))
print(f"[OK] Archive: {zip_path}  ({zip_path.stat().st_size / 1024:.1f} KB)")
print("\nOutputs included:")
for f in sorted(TABLES_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.1f} KB)")

[OK] Archive: /content/task2_outputs_20260514_071037.zip  (50.8 KB)

Outputs included:
  notebook-0  (4.0 KB)
  notebook-1  (4.0 KB)
  notebook_02_trigger_findings.md  (2.3 KB)
  sample_radar_trigger_lane_zone_comparison.csv  (0.3 KB)
  sample_trigger_event_table.csv  (503.6 KB)
  sample_trigger_frequency_summary.csv  (0.5 KB)
  sample_trigger_reference_mapping.csv  (0.2 KB)
  trigger_file_inventory.csv  (32.2 KB)
  trigger_nested_field_paths.csv  (3.5 KB)
  trigger_topic_samples.csv  (0.1 KB)


In [21]:
try:
    from google.colab import files
    print(f"[INFO] Downloading: {zip_path.name}")
    files.download(str(zip_path))
    print("[OK] Download initiated.")
except ImportError:
    print(f"[INFO] Not in Colab — archive at: {zip_path}")

[INFO] Downloading: task2_outputs_20260514_071037.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[OK] Download initiated.


---
## Notebook Complete

### Outputs

| File | Description |
|------|-------------|
| `trigger_file_inventory.csv` | All trigger paths across all 4 zips |
| `trigger_nested_field_paths.csv` | Semantic field locations inside payload |
| `sample_trigger_event_table.csv` | One row per trigger event |
| `sample_trigger_reference_mapping.csv` | `reference_name → lane/zone/sensor` |
| `trigger_topic_samples.csv` | MQTT topic strings with parsed components |
| `sample_trigger_frequency_summary.csv` | Record/event counts per run |
| `sample_radar_trigger_lane_zone_comparison.csv` | Radar vs trigger lane/zone |
| `sample_trigger_radar_alignment_probe.csv` | Timestamp gap statistics |
| `notebook_02_trigger_findings.md` | Auto-generated summary |

### Next: Notebook 03 — Window Dataset Construction

Use trigger `receivedAt` timestamps + radar object records to build
a labelled trigger vs. non-trigger window dataset for all ~400 radar runs.